# Metabolic Health Explorer: modelos iniciales

### Importaciones

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import SplineTransformer, PolynomialFeatures
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint
from sklearn.metrics import roc_auc_score, average_precision_score


RANDOM_STATE = 7

In [2]:
RUTA = Path("..") / "data" / "raw" / "NHANES_consolidado.csv"

df_nhanes = pd.read_csv(RUTA)
adultos = df_nhanes[df_nhanes["Edad"] >= 18].copy()

print("Total de filas", len(df_nhanes))
print("Adultos", len(adultos))
print("Positivos", adultos["Diabetes"].sum())
print("Tasa de positivos", round(adultos["Diabetes"].mean() * 100, 1), "%")

Total de filas 9763
Adultos 6110
Positivos 902
Tasa de positivos 14.8 %


In [3]:
# echamos un ojo a la base
print(df_nhanes.shape)
df_nhanes.head()

(9763, 12)


,Diabetes,Edad,Genero,Raza_Etnia,Ingreso_Hogar,Ingreso_Familia,IMC,Cintura_cm,Presion_Sistolica,Presion_Diastolica,Colesterol_Total,Colesterol_HDL
0,1,69.0,1.0,4.0,4.0,4.0,26.7,100.0,122.0,72.0,167.0,65.0
1,1,54.0,1.0,3.0,7.0,7.0,28.6,107.6,156.0,62.0,170.0,50.0
2,1,72.0,1.0,3.0,10.0,10.0,28.9,109.2,140.0,90.0,126.0,60.0
3,0,9.0,1.0,3.0,9.0,9.0,17.1,61.0,108.0,38.0,168.0,61.0
4,0,73.0,2.0,3.0,15.0,15.0,19.7,NaN,136.0,86.0,201.0,85.0


Cargamos la base consolidada y nos quedamos con los adultos (18 años o más), porque entre los menores solo el 0,5 % tiene diabetes o prediabetes. Quedan 6.110 personas con 902 positivos (14,8 %), igual que en el EDA.

In [4]:
n_ceros = (adultos["Presion_Diastolica"] < 1).sum()
print("Diastólicas en ~0:", n_ceros)

adultos.loc[adultos["Presion_Diastolica"] < 1, "Presion_Diastolica"] = np.nan

# Género: 1 = Hombre, 2 = Mujer en NHANES. Lo pasamos a 0/1 para que sea binaria
adultos["Mujer"] = (adultos["Genero"] == 2).astype(int)

# Porcentaje de nulos por columna
print(adultos.isna().mean().mul(100).round(1).sort_values(ascending=False))

Diastólicas en ~0: 29
Presion_Diastolica    11.7
Presion_Sistolica     11.2
Cintura_cm             8.6
Colesterol_Total       7.7
Colesterol_HDL         7.7
IMC                    4.4
Ingreso_Hogar          1.5
Ingreso_Familia        1.4
Diabetes               0.0
Edad                   0.0
Genero                 0.0
Raza_Etnia             0.0
Mujer                  0.0
dtype: float64


In [5]:
# cómo quedó la limpieza
print('diastólica mínima ahora:', adultos['Presion_Diastolica'].min())
print(adultos['Mujer'].value_counts())
adultos.head()

diastólica mínima ahora: 4.0
Mujer
1    3194
0    2916
Name: count, dtype: int64


,Diabetes,Edad,Genero,Raza_Etnia,Ingreso_Hogar,Ingreso_Familia,IMC,Cintura_cm,Presion_Sistolica,Presion_Diastolica,Colesterol_Total,Colesterol_HDL,Mujer
0,1,69.0,1.0,4.0,4.0,4.0,26.7,100.0,122.0,72.0,167.0,65.0,0
1,1,54.0,1.0,3.0,7.0,7.0,28.6,107.6,156.0,62.0,170.0,50.0,0
2,1,72.0,1.0,3.0,10.0,10.0,28.9,109.2,140.0,90.0,126.0,60.0,0
4,0,73.0,2.0,3.0,15.0,15.0,19.7,NaN,136.0,86.0,201.0,85.0,1
5,0,56.0,1.0,1.0,9.0,9.0,41.7,123.1,160.0,84.0,226.0,38.0,0


Limpieza, igual que en el EDA:
- Presión diastólica casi en 0: los ceros aparecen como `5.4e-79`, por eso se filtran con `< 1` y no con `== 0`. Los pasamos a nulo en vez de botar la fila, para no perder el resto de los datos de esa persona.
- Género: NHANES lo codifica 1 = hombre y 2 = mujer. Lo pasamos a una variable 0/1 que se llama `Mujer`.

In [6]:
VARS_SIN_LAB = ["Edad", "Mujer", "IMC", "Cintura_cm",
                "Presion_Sistolica", "Presion_Diastolica"]

VARS_CON_LAB = VARS_SIN_LAB + ["Colesterol_Total", "Colesterol_HDL"]

OBJETIVO = "Diabetes"

X = adultos[VARS_CON_LAB] 
y = adultos[OBJETIVO]

print(X.shape, y.shape)

(6110, 8) (6110,)


In [7]:
# las dos listas de variables
print('sin lab:', VARS_SIN_LAB)
print('con lab:', VARS_CON_LAB)
X.head()

sin lab: ['Edad', 'Mujer', 'IMC', 'Cintura_cm', 'Presion_Sistolica', 'Presion_Diastolica']
con lab: ['Edad', 'Mujer', 'IMC', 'Cintura_cm', 'Presion_Sistolica', 'Presion_Diastolica', 'Colesterol_Total', 'Colesterol_HDL']


,Edad,Mujer,IMC,Cintura_cm,Presion_Sistolica,Presion_Diastolica,Colesterol_Total,Colesterol_HDL
0,69.0,0,26.7,100.0,122.0,72.0,167.0,65.0
1,54.0,0,28.6,107.6,156.0,62.0,170.0,50.0
2,72.0,0,28.9,109.2,140.0,90.0,126.0,60.0
4,73.0,1,19.7,NaN,136.0,86.0,201.0,85.0
5,56.0,0,41.7,123.1,160.0,84.0,226.0,38.0


Armamos dos escenarios para comparar:
- Sin laboratorio: Edad, Mujer, IMC, Cintura y presión sistólica y diastólica. Son medidas que se toman en cualquier consulta, sin sacar sangre.
- Con laboratorio: las mismas más colesterol total y HDL, que sí necesitan una muestra de sangre.

Raza e ingreso no entran al modelo: sirven para describir a la población, no para decidir a quién priorizar, por equidad. (En el 08 probamos cuánto aportarían, solo para saberlo.)

### Separar entrenamiento y prueba

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,          
    stratify=y,            
    random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape, ", positivos:", y_train.sum(),
      ", tasa:", round(y_train.mean() * 100, 1), "%")
print("Prueba:", X_test.shape,  ", positivos:", y_test.sum(),
      ", tasa:", round(y_test.mean() * 100, 1), "%")

Entrenamiento: (4888, 8) , positivos: 722 , tasa: 14.8 %
Prueba: (1222, 8) , positivos: 180 , tasa: 14.7 %


In [9]:
# revisamos que la partición quedó bien
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))
X_train.head()

(4888, 8) (1222, 8)
Diabetes
0    0.852
1    0.148
Name: proportion, dtype: float64


,Edad,Mujer,IMC,Cintura_cm,Presion_Sistolica,Presion_Diastolica,Colesterol_Total,Colesterol_HDL
5001,80.0,1,24.6,89.5,162.0,74.0,164.0,60.0
3478,80.0,1,31.6,101.8,138.0,70.0,256.0,42.0
5624,56.0,0,29.2,100.3,128.0,62.0,245.0,44.0
7699,40.0,1,33.7,107.9,106.0,66.0,174.0,61.0
2362,60.0,0,29.4,101.6,164.0,76.0,124.0,46.0


Apartamos el 20 % de los datos para la prueba, con partición estratificada (`stratify=y`) para que los dos grupos tengan la misma proporción de positivos (~14,8 %). La prueba no se toca para comparar ni ajustar modelos: se usa una sola vez, al final, con el modelo elegido.



### Métrica de sensibilidad para el tamizaje al 40% de la población

Si solo podemos revisar al 40 % de la población, empezando por los que el modelo ve de mayor riesgo, ¿qué porcentaje de todos los casos encontraríamos?

In [10]:
def recall_at_k(y_true, y_score, k=0.40):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n_tamizar = int(np.ceil(k * len(y_true)))           
    orden = np.argsort(-y_score, kind="stable")         
    top = orden[:n_tamizar]                            

    return y_true[top].sum() / y_true.sum()             

1. Ordenar a las personas de mayor a menor probabilidad.
2. Quedarnos con el 40 % de arriba.
3. Dividir los positivos que hay en ese grupo entre el total de positivos.

In [ ]:
# 1: la edad como puntaje
print("Solo edad:", round(recall_at_k(y, X["Edad"]) * 100, 1), "%")

# 2: puntaje aleatorio
rng = np.random.default_rng(RANDOM_STATE)
print("Aleatorio:", round(recall_at_k(y, rng.random(len(y))) * 100, 1), "%")

# 3: puntaje perfecto
print("Perfecto:", round(recall_at_k(y, y) * 100, 1), "%")

Solo edad: 72.3 %
Aleatorio: 37.5 %
Perfecto: 100.0 %


#### Convertir la métrica en scorer

In [12]:
scorer_r40 = make_scorer(recall_at_k, response_method="predict_proba")

### Modelos de referencia con cross-validation

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

METRICAS = {
    "recall_40": scorer_r40,
    "roc_auc": "roc_auc", 
    "pr_auc": "average_precision",
}

Usamos validación cruzada estratificada de 5 partes sobre el entrenamiento: se entrena con 4 partes y se evalúa en la quinta, rotando. Así sale un resultado por parte y lo resumimos con la media y la desviación estándar.

Métricas:
- recall 40%
- ROC-AUC
- PR-AUC

In [14]:
resultados = [] 

def evaluar(nombre, modelo, columnas):
    cv_res = cross_validate(modelo, X_train[columnas], y_train,
                            cv=cv, scoring=METRICAS)
    fila = {"modelo": nombre, "n_vars": len(columnas)}
    for m in METRICAS:
        fila[f"{m}_media"] = cv_res[f"test_{m}"].mean()
        fila[f"{m}_std"] = cv_res[f"test_{m}"].std()
    resultados.append(fila)
    return pd.DataFrame([fila]).round(3)

#### Modelo Dummy

In [15]:
dummy = DummyClassifier(strategy="prior")
evaluar("Dummy", dummy, VARS_SIN_LAB)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.5,0.0,0.148,0.0


#### Solo edad

In [16]:
solo_edad = LogisticRegression()
evaluar("Solo edad", solo_edad, ["Edad"])

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Solo edad,1,0.724,0.05,0.74,0.022,0.265,0.01


Una regresión logística con una sola variable. Como la probabilidad sube con la edad, ordena igual que ordenar por edad.

In [17]:
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.50,0.000,0.148,0.00
1,Solo edad,1,0.724,0.050,0.74,0.022,0.265,0.01


## Regresión Logística

Armamos un Pipeline con tres pasos:
1. Rellenar nulos con la mediana
2. Estandarizar
3. Regresión logística



In [18]:
def pipeline_logistica(class_weight="balanced", C=1.0):
    return Pipeline([
        ("imputar", SimpleImputer(strategy="median")),   
        ("escalar", StandardScaler()), 
        ("modelo", LogisticRegression(class_weight=class_weight,
                                      C=C, max_iter=1000)),
    ])

In [19]:
# así se ve el pipeline por dentro
pipeline_logistica().steps

[('imputar', SimpleImputer(strategy='median')),
 ('escalar', StandardScaler()),
 ('modelo', LogisticRegression(class_weight='balanced', max_iter=1000))]

In [20]:
evaluar("Logística sin lab", pipeline_logistica(), VARS_SIN_LAB)
evaluar("Logística con lab", pipeline_logistica(), VARS_CON_LAB)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.500,0.000,0.148,0.000
1,Solo edad,1,0.724,0.050,0.740,0.022,0.265,0.010
2,Logística sin lab,6,0.780,0.038,0.790,0.021,0.361,0.054
3,Logística con lab,8,0.798,0.037,0.801,0.021,0.394,0.057


La logística sin laboratorio da un recall 40% de 0,780: por encima de la meta del 75 % y unos 5-6 puntos arriba de la edad sola (0,724). Con laboratorio sube a 0,798, o sea que el colesterol aporta unos 2 puntos más, pero a cambio hay que sacar sangre.

#### Experimento IMC y cintura

In [21]:
sin_cintura = [v for v in VARS_SIN_LAB if v != "Cintura_cm"]
sin_imc     = [v for v in VARS_SIN_LAB if v != "IMC"]

evaluar("Logística sin lab - solo IMC",    pipeline_logistica(), sin_cintura)
evaluar("Logística sin lab - solo cintura", pipeline_logistica(), sin_imc)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.500,0.000,0.148,0.000
1,Solo edad,1,0.724,0.050,0.740,0.022,0.265,0.010
2,Logística sin lab,6,0.780,0.038,0.790,0.021,0.361,0.054
3,Logística con lab,8,0.798,0.037,0.801,0.021,0.394,0.057
4,Logística sin lab - solo IMC,5,0.778,0.036,0.784,0.019,0.341,0.044
5,Logística sin lab - solo cintura,5,0.778,0.045,0.789,0.022,0.361,0.053


IMC y cintura tienen una correlación de 0,91 en el EDA, así que probamos quedarnos con una sola de las dos.

Las tres versiones dan prácticamente lo mismo (~0,778-0,780), dentro de lo que varía la validación cruzada. Las dos variables dicen casi lo mismo. La cintura es un poquito más informativa y tiene un coeficiente mayor.

#### Experimento class_weight

In [22]:
evaluar("Logística sin lab - sin class_weight",
        pipeline_logistica(class_weight=None), VARS_SIN_LAB)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.500,0.000,0.148,0.000
1,Solo edad,1,0.724,0.050,0.740,0.022,0.265,0.010
2,Logística sin lab,6,0.780,0.038,0.790,0.021,0.361,0.054
3,Logística con lab,8,0.798,0.037,0.801,0.021,0.394,0.057
4,Logística sin lab - solo IMC,5,0.778,0.036,0.784,0.019,0.341,0.044
5,Logística sin lab - solo cintura,5,0.778,0.045,0.789,0.022,0.361,0.053
6,Logística sin lab - sin class_weight,6,0.781,0.038,0.790,0.021,0.366,0.053


La diferencia es mínima (0,780 frente a 0,781). Los pesos de clase suben todas las probabilidades, pero casi no cambian el orden de las personas, que es lo único que mide el recall@40 %.

In [23]:
modelo_sin = pipeline_logistica().fit(X_train[VARS_SIN_LAB], y_train)
modelo_con = pipeline_logistica().fit(X_train[VARS_CON_LAB], y_train)

coefs = pd.DataFrame({
    "sin_lab": pd.Series(modelo_sin.named_steps["modelo"].coef_[0], index=VARS_SIN_LAB),
    "con_lab": pd.Series(modelo_con.named_steps["modelo"].coef_[0], index=VARS_CON_LAB),
}).round(3)
coefs.sort_values("con_lab", key=abs, ascending=False)

,sin_lab,con_lab
Edad,0.942,0.963
Cintura_cm,0.573,0.500
Colesterol_HDL,NaN,-0.302
Colesterol_Total,NaN,-0.219
Presion_Sistolica,0.153,0.178
IMC,0.152,0.141
Mujer,0.028,0.129
Presion_Diastolica,-0.026,0.015


- Edad (~0,95) y cintura (~0,5-0,6) son las que más pesan.
- El IMC queda con un coeficiente chiquito porque la cintura ya le quitó casi toda su información (se parecen mucho). 
- La presión diastólica queda cerca de 0, así que es candidata a salir.
- HDL negativo: el HDL alto protege, el signo es el esperado.

### Regresión Logística con no linealidades

La logística supone que el riesgo sube en línea recta con cada variable. Pero en el EDA vimos que con la edad no es así (1,6 % en 18-29 años y ~30 % en 60-69).

Entonces probamos splines (SplineTransformer), que parten la variable en tramos y ajustan una curva suave en cada uno. Se los pusimos a Edad y Cintura, las dos que más pesaban en los coeficientes. El ColumnTransforme nos deja tratar distinto a cada grupo de columnas: splines a unas y estandarización a las otras.

In [24]:
def pipeline_splines(columnas, vars_spline=("Edad", "Cintura_cm"),
                     interaccion=False, C=1.0, n_knots=5):
    vars_spline = list(vars_spline)
    vars_lineales = [c for c in columnas if c not in vars_spline]

    transformaciones = [
        ("splines", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(n_knots=n_knots, degree=3)),
        ]), vars_spline),

        ("lineales", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), vars_lineales),
    ]

    if interaccion:
        transformaciones.append(
            ("interaccion", Pipeline([
                ("imputar", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler()),
                ("producto", PolynomialFeatures(degree=2, interaction_only=True,
                                                include_bias=False)),
            ]), ["Edad", "Cintura_cm"])
        )

    return Pipeline([
        ("preparar", ColumnTransformer(transformaciones)),
        ("modelo", LogisticRegression(class_weight="balanced", C=C, max_iter=2000)),
    ])

In [25]:
# cuántas columnas le salen a cada pipeline de splines antes de la regresión
for nombre, p in [('splines sin lab', pipeline_splines(VARS_SIN_LAB)),
                  ('splines + interacción sin lab', pipeline_splines(VARS_SIN_LAB, interaccion=True))]:
    p.fit(X_train[VARS_SIN_LAB], y_train)
    print(nombre, '->', p.named_steps['preparar'].transform(X_train[VARS_SIN_LAB]).shape[1], 'columnas')

splines sin lab -> 18 columnas
splines + interacción sin lab -> 21 columnas


In [26]:
evaluar("Splines sin lab", pipeline_splines(VARS_SIN_LAB), VARS_SIN_LAB)
evaluar("Splines + interacción sin lab",
        pipeline_splines(VARS_SIN_LAB, interaccion=True), VARS_SIN_LAB)
evaluar("Splines + interacción con lab",
        pipeline_splines(VARS_CON_LAB, interaccion=True), VARS_CON_LAB)

pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.500,0.000,0.148,0.000
1,Solo edad,1,0.724,0.050,0.740,0.022,0.265,0.010
2,Logística sin lab,6,0.780,0.038,0.790,0.021,0.361,0.054
3,Logística con lab,8,0.798,0.037,0.801,0.021,0.394,0.057
4,Logística sin lab - solo IMC,5,0.778,0.036,0.784,0.019,0.341,0.044
5,Logística sin lab - solo cintura,5,0.778,0.045,0.789,0.022,0.361,0.053
6,Logística sin lab - sin class_weight,6,0.781,0.038,0.790,0.021,0.366,0.053
7,Splines sin lab,6,0.788,0.040,0.797,0.025,0.375,0.053
8,Splines + interacción sin lab,6,0.783,0.041,0.796,0.024,0.373,0.050
9,Splines + interacción con lab,8,0.801,0.033,0.809,0.023,0.413,0.051


Los splines mejoran un poco el ROC-AUC (de 0,790 a 0,796-0,797) y el PR-AUC (de 0,361 a 0,373-0,375), y el recall 40% pasa de 0,780 a 0,783-0,788, una diferencia menor que la desviación (~0,04).

Pasa porque el ROC-AUC mira el orden de toda la lista, mientras que el recall 40% solo mira si cada caso queda arriba o abajo del corte del 40%. Los splines ordenan mejor a la gente que está lejos del corte, pero casi no mueven a nadie de un lado al otro. La interacción Edad × Cintura tampoco ayuda (0,783 frente a 0,788 sin ella): en escala log-odds el efecto conjunto ya queda como suma de efectos.

#### Ajuste de hiperparámetros

In [27]:
grid = GridSearchCV(
    pipeline_splines(VARS_SIN_LAB, interaccion=True),
    param_grid={
        "modelo__C": [0.01, 0.1, 1, 10],
        "preparar__splines__spline__n_knots": [3, 4, 5, 7],
    },
    scoring=scorer_r40,
    cv=cv,
)
grid.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor recall@40 %:", round(grid.best_score_, 3))

(pd.DataFrame(grid.cv_results_)
   [["params", "mean_test_score", "std_test_score"]]
   .sort_values("mean_test_score", ascending=False)
   .head(6))

Mejores parámetros: {'modelo__C': 10, 'preparar__splines__spline__n_knots': 4}
Mejor recall@40 %: 0.792


,params,mean_test_score,std_test_score
13,"{'modelo__C': 10, 'preparar__splines__spline__...",0.792308,0.039562
12,"{'modelo__C': 10, 'preparar__splines__spline__...",0.790920,0.042498
3,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.790910,0.034870
9,"{'modelo__C': 1, 'preparar__splines__spline__n...",0.788151,0.039970
0,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.788142,0.036516
4,"{'modelo__C': 0.1, 'preparar__splines__spline_...",0.788142,0.029202


Con GridSearchCV ajustamos la fuerza de la regularización (C; menor valor = más regularización) y el número de nodos de los splines (n_knots: más nodos = curva más flexible), usando el recall 40% como criterio. La mejor combinación fue C = 10 y 4 nodos (recall 0,792), pero las seis mejores quedan a menos de 0,004 una de otra, así que el ajuste casi no importa.

## Modelos de árboles

- Random Forest: cientos de árboles independientes, cada uno con una muestra distinta, y se promedian.
- HistGradientBoosting (HGB): árboles en secuencia, cada uno corrige los errores del anterior.


### Sin ajuste

In [28]:
hgb_default = HistGradientBoostingClassifier(class_weight="balanced",
                                             random_state=RANDOM_STATE)

rf_default = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),   
    ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                      random_state=RANDOM_STATE, n_jobs=-1)),
])

for nombre, modelo in [("HGB por defecto", hgb_default), ("RF por defecto", rf_default)]:
    res = cross_validate(modelo, X_train[VARS_SIN_LAB], y_train, cv=cv,
                         scoring=METRICAS, return_train_score=True)
    print(f"{nombre}: recall@40 % entrenamiento = {res['train_recall_40'].mean():.3f}, "
          f"validación = {res['test_recall_40'].mean():.3f}")


HGB por defecto: recall@40 % entrenamiento = 1.000, validación = 0.741


RF por defecto: recall@40 % entrenamiento = 1.000, validación = 0.740


Diagnóstico de sobreajuste: con los parámetros por defecto, los dos logran un recall 40% perfecto (1,000) en entrenamiento pero solo 0,741 (HGB) y 0,740 (RF) en validación, por debajo de la regresión logística (0,780). Los árboles crecen hasta separar casi a cada persona y se aprenden el ruido de memoria. Toca limitarlos ajustando hiperparámetros.

### Ajustar HistGradientBoosting

In [29]:
busqueda_hgb = RandomizedSearchCV(
    HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    param_distributions={
        "learning_rate":     loguniform(0.01, 0.3),   
        "max_iter":          randint(50, 400),        
        "max_depth":         [2, 3, 4, None],         
        "min_samples_leaf":  randint(20, 200),        
        "l2_regularization": loguniform(1e-3, 10),    
    },
    n_iter=30,              
    scoring=scorer_r40,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
busqueda_hgb.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", busqueda_hgb.best_params_)
print("Mejor recall@40 %:", round(busqueda_hgb.best_score_, 3))

Mejores parámetros: {'l2_regularization': np.float64(0.024832789239950134), 'learning_rate': np.float64(0.025612133220084665), 'max_depth': 2, 'max_iter': 367, 'min_samples_leaf': 155}
Mejor recall@40 %: 0.787


Usamos RandomizedSearchCV, que prueba 30 combinaciones sorteadas en vez de una grilla completa (con 5 hiperparámetros y 4 valores cada uno serían 1.024 combinaciones).

### Ajustar Random Forest

In [30]:
busqueda_rf = RandomizedSearchCV(
    rf_default,
    param_distributions={
        "modelo__max_depth":        [3, 4, 5, 6, 8, None],
        "modelo__min_samples_leaf": randint(5, 100),
        "modelo__max_features":     ["sqrt", 0.5, None],  
    },
    n_iter=20,
    scoring=scorer_r40,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
busqueda_rf.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", busqueda_rf.best_params_)
print("Mejor recall@40 %:", round(busqueda_rf.best_score_, 3))


Mejores parámetros: {'modelo__max_depth': 8, 'modelo__max_features': 'sqrt', 'modelo__min_samples_leaf': 12}
Mejor recall@40 %: 0.79


Ajustamos la profundidad máxima, el tamaño mínimo de hoja y cuántas variables mira cada corte. El mejor usa max_depth = 8 y min_samples_leaf = 12, más restringido que el modelo por defecto (que crece sin límite), aunque menos que el HGB.

### Agregar a tabla de resultados

In [31]:
evaluar("HGB ajustado sin lab", busqueda_hgb.best_estimator_, VARS_SIN_LAB)
evaluar("RF ajustado sin lab",  busqueda_rf.best_estimator_,  VARS_SIN_LAB)

hgb_con_lab = HistGradientBoostingClassifier(class_weight="balanced",
                                             random_state=RANDOM_STATE,
                                             **busqueda_hgb.best_params_)
evaluar("HGB ajustado con lab", hgb_con_lab, VARS_CON_LAB)

pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.421,0.024,0.500,0.000,0.148,0.000
1,Solo edad,1,0.724,0.050,0.740,0.022,0.265,0.010
2,Logística sin lab,6,0.780,0.038,0.790,0.021,0.361,0.054
3,Logística con lab,8,0.798,0.037,0.801,0.021,0.394,0.057
4,Logística sin lab - solo IMC,5,0.778,0.036,0.784,0.019,0.341,0.044
5,Logística sin lab - solo cintura,5,0.778,0.045,0.789,0.022,0.361,0.053
6,Logística sin lab - sin class_weight,6,0.781,0.038,0.790,0.021,0.366,0.053
7,Splines sin lab,6,0.788,0.040,0.797,0.025,0.375,0.053
8,Splines + interacción sin lab,6,0.783,0.041,0.796,0.024,0.373,0.050
9,Splines + interacción con lab,8,0.801,0.033,0.809,0.023,0.413,0.051


Ya ajustados, los árboles llegan a 0,787 (HGB) y 0,790 (RF) sin laboratorio, un poquito por encima de la regresión logística (0,780), pero la diferencia es menor que la desviación estándar. 

## Comparación y elección de modelo

In [32]:
tabla = (pd.DataFrame(resultados)
           .drop_duplicates(subset="modelo", keep="last")     
           .sort_values("recall_40_media", ascending=False)
           .reset_index(drop=True))

tabla["supera_meta"] = tabla["recall_40_media"] >= 0.75
tabla.round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std,supera_meta
0,Splines + interacción con lab,8,0.801,0.033,0.809,0.023,0.413,0.051,True
1,Logística con lab,8,0.798,0.037,0.801,0.021,0.394,0.057,True
2,RF ajustado sin lab,6,0.790,0.034,0.791,0.023,0.380,0.039,True
3,Splines sin lab,6,0.788,0.040,0.797,0.025,0.375,0.053,True
4,HGB ajustado sin lab,6,0.787,0.030,0.788,0.023,0.370,0.038,True
5,Splines + interacción sin lab,6,0.783,0.041,0.796,0.024,0.373,0.050,True
6,Logística sin lab - sin class_weight,6,0.781,0.038,0.790,0.021,0.366,0.053,True
7,HGB ajustado con lab,8,0.780,0.033,0.803,0.020,0.420,0.034,True
8,Logística sin lab,6,0.780,0.038,0.790,0.021,0.361,0.054,True
9,Logística sin lab - solo IMC,5,0.778,0.036,0.784,0.019,0.341,0.044,True


In [33]:
mejor = tabla.loc[0]
print(f"Mejor modelo: {mejor['modelo']} → {mejor['recall_40_media']:.3f} ± {mejor['recall_40_std']:.3f}")

sin_lab = tabla[tabla["n_vars"] <= 6]
mejor_sin = sin_lab.iloc[0]
umbral = mejor_sin["recall_40_media"] - mejor_sin["recall_40_std"]

print(f"\nMejor sin lab: {mejor_sin['modelo']} ({mejor_sin['recall_40_media']:.3f})")
print(f"Modelos sin lab estadísticamente empatados (≥ {umbral:.3f}):")
print(sin_lab[sin_lab["recall_40_media"] >= umbral][["modelo", "recall_40_media", "recall_40_std"]].round(3))

Mejor modelo: Splines + interacción con lab → 0.801 ± 0.033

Mejor sin lab: RF ajustado sin lab (0.790)
Modelos sin lab estadísticamente empatados (≥ 0.756):
                                  modelo  recall_40_media  recall_40_std
2                    RF ajustado sin lab            0.790          0.034
3                        Splines sin lab            0.788          0.040
4                   HGB ajustado sin lab            0.787          0.030
5          Splines + interacción sin lab            0.783          0.041
6   Logística sin lab - sin class_weight            0.781          0.038
8                      Logística sin lab            0.780          0.038
9           Logística sin lab - solo IMC            0.778          0.036
10      Logística sin lab - solo cintura            0.778          0.045


Las ocho variantes sin laboratorio dan un recall 40% entre 0,778 y 0,790. La diferencia entre la mejor y la peor (0,012) es menor que la desviación estándar de cualquiera (0,030-0,045), así que con estos datos no se pueden distinguir. Aplicando la regla de una desviación estándar, escogemos el más simple de los empatados.

Modelo elegido: regresión logística sin laboratorio.

El escenario con laboratorio le gana al otro por unos 2 puntos en la regresión logística (0,798 frente a 0,780), pero hay que sacar sangre y el coeficiente del colesterol total seguramente refleja el tratamiento y no el riesgo.

#### Entrenar todo el modelo y evaluarlo

In [34]:
# 1. Entrenar con TODO el conjunto de entrenamiento
modelo_final = pipeline_logistica().fit(X_train[VARS_SIN_LAB], y_train)
referencia_edad = LogisticRegression().fit(X_train[["Edad"]], y_train)

# 2. Probabilidades en prueba
prob_final = modelo_final.predict_proba(X_test[VARS_SIN_LAB])[:, 1]
prob_edad = referencia_edad.predict_proba(X_test[["Edad"]])[:, 1]

# 3. Métricas
eval_prueba = pd.DataFrame({
    "modelo": ["Logística sin lab (final)", "Solo edad (referencia)"],
    "recall_40": [recall_at_k(y_test, prob_final), recall_at_k(y_test, prob_edad)],
    "roc_auc": [roc_auc_score(y_test, prob_final), roc_auc_score(y_test, prob_edad)],
    "pr_auc": [average_precision_score(y_test, prob_final), 
               average_precision_score(y_test, prob_edad)],
}).round(3)
eval_prueba

,modelo,recall_40,roc_auc,pr_auc
0,Logística sin lab (final),0.750,0.781,0.340
1,Solo edad (referencia),0.728,0.747,0.265


In [35]:
# ¿cuántas personas y cuántos positivos hay en la prueba?
print('prueba:', len(y_test), 'personas |', int(y_test.sum()), 'positivos')

prueba: 1222 personas | 180 positivos


Reentrenamos el modelo elegido con todo el entrenamiento y lo evaluamos una sola vez en la prueba, junto con la regla de solo edad sobre las mismas personas, para que la comparación sea justa.

### Intervalo de confianza con Bootstrap

Con 180 positivos en la prueba, el resultado puede moverse varios puntos según quién haya caído ahí. El bootstrap simula 2.000 pruebas distintas remuestreando con reemplazo y calcula la métrica en cada una. El 95 % central es el intervalo de confianza.

In [36]:
rng = np.random.default_rng(RANDOM_STATE)
y_t = y_test.to_numpy()
n = len(y_t)

recall_final, diferencia = [], []
for _ in range(2000):
    idx = rng.integers(0, n, n)                       
    r_final = recall_at_k(y_t[idx], prob_final[idx])
    r_edad = recall_at_k(y_t[idx], prob_edad[idx])
    recall_final.append(r_final)
    diferencia.append(r_final - r_edad)

ic_final = np.percentile(recall_final, [2.5, 97.5])
ic_dif = np.percentile(diferencia, [2.5, 97.5])

print(f"recall@40 % del modelo final: IC 95 % = [{ic_final[0]:.3f}, {ic_final[1]:.3f}]")
print(f"Mejora sobre la edad: IC 95 % = [{ic_dif[0]:.3f}, {ic_dif[1]:.3f}]")
print(f"% de remuestreos en que el modelo supera a la edad: {np.mean(np.array(diferencia) > 0):.1%}")


recall@40 % del modelo final: IC 95 % = [0.698, 0.823]
Mejora sobre la edad: IC 95 % = [-0.026, 0.093]
% de remuestreos en que el modelo supera a la edad: 81.6%


Meta de negocio: el modelo captura el 75,0 % de los casos revisando al 40 % (IC 95 % bootstrap: 69,8 %-82,3 %). Cae justo en la meta y el límite de abajo del intervalo queda por debajo del 75 %, así que con esta partición no podemos asegurar que se cumpla.

Comparación con la edad sola: el modelo la supera por 2,2 puntos (0,750 frente a 0,728), pero la mejora es positiva en el 81,6 % de los remuestreos y su intervalo del 95 % (de -2,6 a 9,3 puntos) incluye el cero. 
